Bringing over the same set up as my base model (which will be ran last for comparison)

In [3]:
import pickle
import pandas as pd

df = pd.read_pickle('../data/cleaned_games.pkl')

with open('../data/genre_columns.pkl', 'rb') as f:
    genre_features = pickle.load(f)

features = [
    'price',
    'year',
    'num_tags',
    'dev_success',
    'dev_had_success',
    'log_dev_experience',
    'num_devs',
] + list(genre_features)

X = df[features]
y = df['hit']

split_index = int(len(df) * 0.7)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]



Now, with this as a basis, I will complete some model analysis before moving into my second layer, the main purpose of this project. I will start by separating th efeatures into subgroups to see if any one group is entirely carrying the model.

In [4]:
dev_features = [
    'dev_success',
    'dev_had_success',
    'log_dev_experience',
    'num_devs'
]

meta_features = ['num_tags']

genre_features = list(genre_features)

other_features = ['price', 'year']

features_no_tags = [f for f in features if f != 'num_tags']

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def run_model(feature_list):
    X_train_sub = X_train[feature_list]
    X_test_sub = X_test[feature_list]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sub)
    X_test_scaled = scaler.transform(X_test_sub)
    
    model = LogisticRegression(max_iter=3000, class_weight='balanced')
    model.fit(X_train_scaled, y_train)
    
    y_probs = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_probs > 0.35).astype(int)

    report = classification_report(y_test, y_pred, output_dict=True)
    f1 = report['1']['f1-score']
    precision = report['1']['precision']
    recall = report['1']['recall']
    
    cm = confusion_matrix(y_test, y_pred)
    
    # clean output
    result = {
        "F1": round(f1, 3),
        "Precision": round(precision, 3),
        "Recall": round(recall, 3),
        "Confusion Matrix": cm
    }
    
    return result

In [5]:
run_model(dev_features)

{'F1': 0.074,
 'Precision': 0.038,
 'Recall': 0.923,
 'Confusion Matrix': array([[ 6742, 26327],
        [   88,  1054]])}

In [6]:
run_model(genre_features)

{'F1': 0.073,
 'Precision': 0.038,
 'Recall': 0.906,
 'Confusion Matrix': array([[ 6969, 26100],
        [  107,  1035]])}

In [7]:
run_model(meta_features)

{'F1': 0.152,
 'Precision': 0.082,
 'Recall': 0.994,
 'Confusion Matrix': array([[20404, 12665],
        [    7,  1135]])}

In [8]:
run_model(features)

{'F1': 0.302,
 'Precision': 0.224,
 'Recall': 0.461,
 'Confusion Matrix': array([[31247,  1822],
        [  615,   527]])}

In [ ]:
features_clean = [
    'price',
    'year',
    'dev_success',
    'dev_had_success',
    'log_dev_experience',
    'num_devs'
] + list(genre_features)

In [23]:

X_train_sub = X_train[features_clean]
X_test_sub = X_test[features_clean]
    
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sub)
X_test_scaled = scaler.transform(X_test_sub)
    
model = LogisticRegression(max_iter=3000, class_weight='balanced')
model.fit(X_train_scaled, y_train)
    
y_probs = model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_probs > 0.5).astype(int)

report = classification_report(y_test, y_pred, output_dict=True)
f1 = report['1']['f1-score']
precision = report['1']['precision']
recall = report['1']['recall']
    
cm = confusion_matrix(y_test, y_pred)
    

result = {
    "F1": round(f1, 3),
    "Precision": round(precision, 3),
    "Recall": round(recall, 3),
    "Confusion Matrix": cm
}

print(result)

{'F1': 0.212, 'Precision': 0.194, 'Recall': 0.234, 'Confusion Matrix': array([[31959,  1110],
       [  875,   267]])}
